In [3]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('pandas_orders.csv', parse_dates=['order_date', 'signup_date'])
df.head(1)

,order_id,user_id,user_name,segment,signup_date,home_city,city,region,order_date,channel,product,category,price,quantity,discount_pct,order_total_rub,status,delivery_days,promo_code
0,50387,22,Егор Соколова,regular,2024-05-21,Москва,Москва,Центр,2026-01-02,marketplace,Наушники,Техника,6030.0,1,5,5728.5,completed,6.0,WELCOME


In [4]:
df['revenue'] = df['price'] * df['quantity']
df[['order_id', 'city', 'order_date', 'quantity' ]][df['status']=='completed']

,order_id,city,order_date,quantity
0,50387,Москва,2026-01-02,1
1,50480,Москва,2026-01-02,1
2,50309,Краснодар,2026-01-03,1
3,50234,Санкт-Петербург,2026-01-04,1
4,50282,Казань,2026-01-04,1
...,...,...,...,...
502,50417,Новосибирск,2026-09-27,4
503,50195,Екатеринбург,2026-09-28,1
504,50069,Москва,2026-09-29,2
505,50082,Москва,2026-09-29,1


In [6]:
df[df['status']=='completed'].sort_values("revenue", ascending=False).head(5)


,order_id,user_id,user_name,segment,signup_date,home_city,city,region,order_date,channel,product,category,price,quantity,discount_pct,order_total_rub,status,delivery_days,promo_code,revenue
448,50092,48,Степан Иванова,regular,2024-09-25,Новосибирск,Новосибирск,Сибирь,2026-08-28,app,Ноутбук,Техника,68440.0,3,0,205320.0,completed,7.0,NaN,205320.0
357,50020,47,Рита Кузнецов,regular,2024-02-27,Москва,Екатеринбург,Урал,2026-07-15,website,Ноутбук,Техника,61640.0,3,0,184920.0,completed,7.0,NaN,184920.0
304,50060,38,Егор Егорова,regular,2025-03-05,Екатеринбург,Екатеринбург,Урал,2026-06-16,app,Планшет,Техника,29840.0,3,0,89520.0,completed,6.0,NaN,89520.0
128,50399,42,Лев Фёдорова,new,2025-07-27,Казань,Москва,Центр,2026-03-10,website,Планшет,Техника,28840.0,3,0,86520.0,completed,3.0,NaN,86520.0
91,50408,26,Лев Егорова,regular,2024-12-19,Казань,Казань,Поволжье,2026-02-19,app,Ноутбук,Техника,68930.0,1,15,58590.5,completed,1.0,VIP,68930.0


In [10]:
df_agg = (df[df['status']=='completed'].groupby('city')
          .agg(
              sum_revenue = ('revenue', 'sum'),
              cnt = ('status', 'count')
          )
)
df_agg

,sum_revenue,cnt
city,,
Екатеринбург,472100.0,49
Казань,438820.0,69
Краснодар,279770.0,70
Москва,404880.0,80
Новосибирск,455890.0,59
Санкт-Петербург,168320.0,63


In [11]:
df_agg = (df[df['status']=='completed'].groupby('region')
          .agg(
              sum_revenue = ('revenue', 'sum'))
)
df_agg

,sum_revenue
region,
Поволжье,438820.0
Северо-Запад,168320.0
Сибирь,455890.0
Урал,472100.0
Центр,404880.0
Юг,279770.0


In [12]:
order_facts = df.drop(columns=["user_name", "segment", "signup_date", "home_city", "region"])

known_users = df.dropna(subset=["user_name"])  # Оставляю заказы с заполненной карточкой клиента
# Беру поля карточки
users = known_users[["user_id", "user_name", "segment", "signup_date", "home_city"]]
# Оставляю одну карточку на клиента, а не на каждый заказ
users = users.drop_duplicates(subset="user_id")

# Для справочника нужны и город, и его регион
known_cities = df.dropna(subset=["city", "region"])
regions = known_cities[["city", "region"]]  # Оставляю только поля справочника регионов
regions = regions.drop_duplicates(subset="city")  # Оставляю одну строку на город

In [15]:
 
df_segments = (order_facts[order_facts["status"] == "completed"]
                 .merge(users[["user_id", "segment"]] , on="user_id", how="inner")
                 .groupby("segment", as_index=False)
                 .agg(revenue=("order_total_rub", "sum"))
                 .sort_values("revenue", ascending=False)
                 )
df_segments


,segment,revenue
1,regular,1652980.5
0,new,331005.5
2,vip,175666.0


In [17]:
df_agg = (df[df['status']=='completed'].groupby('city')
          .agg(sum_revenue = ('revenue', 'sum'))
          .sort_values("sum_revenue", ascending=False)
)
df_agg

,sum_revenue
city,
Екатеринбург,472100.0
Новосибирск,455890.0
Казань,438820.0
Москва,404880.0
Краснодар,279770.0
Санкт-Петербург,168320.0


In [18]:
df_agg = (df[df['status']=='completed'].groupby('city')
          .agg(
              sum_revenue = ('revenue', 'sum'),
              cnt = ('status', 'count'),
              users = ('user_id', 'nunique')
          )
)
df_agg[df_agg['users'] > 30]

,sum_revenue,cnt,users
city,,,
Казань,438820.0,69,34
Краснодар,279770.0,70,32
Москва,404880.0,80,39


In [19]:
completed = df[df["status"] == "completed"].copy()
completed["month"] = completed["order_date"].dt.to_period("M")
monthly_revenue = completed.groupby(["month", "city"], as_index=False).agg(revenue=("order_total_rub", "sum"))  
monthly_revenue.sort_values(["month", "city"])

,month,city,revenue
0,2026-01,Екатеринбург,13184.0
1,2026-01,Казань,7835.0
2,2026-01,Краснодар,31250.0
3,2026-01,Москва,18693.5
4,2026-01,Новосибирск,15723.0
5,2026-01,Санкт-Петербург,6412.0
6,2026-02,Екатеринбург,7060.0
7,2026-02,Казань,175994.5
8,2026-02,Краснодар,43488.0
9,2026-02,Москва,70128.0


In [20]:
category_report = (df[df["status"] == "completed"]
                   .groupby("category", as_index=False)
                   .agg(  # Собираю строку на категорию
    orders_count=("order_id", "count"),  # Считаю завершенные заказы
    revenue=("order_total_rub", "sum"),  # Складываю суммы этих заказов
                   ).sort_values("revenue", ascending=False)

)
category_report  

,category,orders_count,revenue
4,Техника,56,1698292.5
1,Дом,57,184912.5
3,Напитки,180,130889.5
0,Аксессуары,53,84700.0
2,Книги,47,71036.5


In [21]:
clean_orders = order_facts.dropna(subset=["city", "order_total_rub"])
completed = clean_orders[clean_orders["status"] == "completed"]  
user_segments = users[["user_id", "segment"]] 
# Сохраняю и заказы без карточки клиента
joined = completed.merge(user_segments, on="user_id", how="left")
city_report = joined.groupby("city", as_index=False).agg(  
    orders_count=("order_id", "count"),  
    users=("user_id", "nunique"),  
    revenue=("order_total_rub", "sum"), 
)  
city_report.sort_values("revenue", ascending=False)  

,city,orders_count,users,revenue
0,Екатеринбург,49,26,471301.0
4,Новосибирск,59,24,440777.5
1,Казань,69,34,420703.0
3,Москва,80,39,397008.0
2,Краснодар,69,32,274392.5
5,Санкт-Петербург,62,29,162826.5
